<div align="center">
  <h1></h1>
  <h1>Retrieval-Augmented Generation</h1>
  <h4 align="center">Assignmnet II</h4>
</div>

# Part 1 – Foundational Concepts

Welcome to Assignment II! In this notebook, you will build and implement a Retrieval-Augmented Generation (RAG) pipeline.



### Please answer the following questions in short. Avoid using llms to answer this part!

1. Define RAG and describe how it improves generative model responses compared to an LLM without retrieval.

RAG(Retrieval-Augmented Generation) is a technique that combines information retrieval with language generation. It works by first retrieving relevant documents from an external knowledge base, then using those documents as context to ground the LLM's response. This improves responses compared to standalone LLMs because it provides factual,up-to-date information from external sources rather than relying soley on the model's training data, which may be outdated or incomplete.


2. List and briefly explain the core stages of a RAG pipeline (indexing, retrieval, augmentation, generation).


The Four core stages are :

 1. Indexing: Documents are processed , split into chunks, converted to embeddings, and stored in a vector database for efficient retrieval

 2. Retrieval : When a query arrives, relevant document chunks are retrieved fro the vector store based on semantic similarity.

 3. Augmentation : Retrieved documents are formatted and combined with the user's question to create an enriched prompt .

 4. Generation : The LLM generates an answer using both the original question and the retrieved context.


3. Explain why RAG can reduce “hallucinations” compared to plain generative models — and why it doesn’t eliminate them entirely.


RAG reduces hallucinations by grounding the models' responses in retrieved factual documents , giving the LLM concrete information to reference rather than generating answers purely from its parametric memory. However, it does not eliminate hallucinations entirely because i. The retrieval system might fetch irrelevant or incorrect documents . ii. The LLM might still misinterpret or misuse the provided context , and iii . The source documents themselves might contain errors or biases .

4. Compare two retrieval methods (e.g., BM25 vs Dense Embeddings) and discuss how they affect RAG performance.

BM25 :
A lexical retrieval method based on keyword matching, using term frequency and inverse document frequency. It excels when exact terminoloy matters and works well for queries with specific technical terms. However, it cannot understand semantic similarity.

Dense Embeddings :
Transform text into vector representations that capture semantic meaning. They excel at finding conceptually similar documents even when different words are used, but may miss exact keyword matches. Dense embeddings are better for semantic understanding while BM25 is better for precision-based keyword retrieval

# Part 2 – Implementation

# 2.1. Access to Groq
Execute the following cell to connect to your Groq account.

In [221]:
import os
os.environ["GROQ_API_KEY"] = "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"


# 2.2. Packages  
Execute the following code cells for installing the packages needed.

note: If there are package conflics you can use pip-tools to automatically find and install the compatible versions. If you won't be using specific libraries that can't be installed, you can ignore them.

In [222]:
!pip install -q requests==2.32.4
!pip install -q langchain langchain-community chromadb sentence-transformers python-dotenv


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
google-adk 1.21.0 requires opentelemetry-api<=1.37.0,>=1.37.0, but you have opentelemetry-api 1.39.1 which is incompatible.
google-adk 1.21.0 requires opentelemetry-sdk<=1.37.0,>=1.37.0, but you have opentelemetry-sdk 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-exporter-otlp-proto-common==1.37.0, but you have opentelemetry-exporter-otlp-proto-common 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-proto==1.37.0, but you have opentelemetry-proto 1.39.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.37.0 requires opentelemetry-sdk~=1.37.0, but you have opentelemetry-sdk 1.39.1 which is 

In [223]:
import requests
import json

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


In [224]:
!pip install -q groq langchain-groq beautifulsoup4


# 2.3. LLM Interface (Groq API)

This cell defines a simple, reusable interface for interacting with a large language model hosted by Groq. It initializes the Groq client using an API key from the environment and provides a helper function (call_groq_llm) that sends a prompt to a specified Groq model and returns the generated response.

In [225]:
import os
from groq import Groq

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

def call_groq_llm(prompt, model="llama-3.1-8b-instant"):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


# 2.4. Document Preparation and Chunking

In this cell, you will prepare the knowledge base used by the RAG system. Your task is to define a small collection of documents that represent external knowledge and then split these documents into smaller, overlapping chunks suitable for retrieval. Proper document selection and chunking are crucial, as they directly affect retrieval quality and, consequently, the final generated answers.

In [226]:
# TODO:
# 1. Extend the `docs` list with more Document objects related to RAG.
# 2. Make sure the documents contain meaningful explanatory text (not just one sentence).
# 3. Optionally adjust `chunk_size` and `chunk_overlap` and justify your choice.

docs = [
    Document(page_content="Retrieval Augmented Generation (RAG) helps ground generation in external knowledge."),
    Document(page_content="Advanced techniques like Self-RAG and RRR improve answer quality using refinement loops."),
    Document(page_content="RAG combines the power of large language models with the ability to retrieve relevant information from external knowledge bases. This approach significantly reduces hallucinations and improves factual accuracy."),
    Document(page_content="The core components of a RAG system include document indexing, semantic retrieval using vector embeddings, context construction, and grounded generation. Each component plays a crucial role in ensuring high-quality outputs."),
    Document(page_content="Vector databases like Chroma enable efficient semantic search by storing document embeddings in high-dimensional space. Queries are embedded using the same model, and cosine similarity is used to find the most relevant chunks."),
    Document(page_content="Self-RAG introduces reflection tokens that allow the model to critique and refine its own outputs. This self-reflection mechanism helps identify when retrieved documents are not relevant or when the generated answer needs improvement."),
    Document(page_content="RRR-RAG (Rewrite-Retrieve-Respond) improves retrieval quality by first rewriting the user query to be more retrieval-friendly, then fetching documents, and finally generating an answer based on the retrieved context."),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
split_docs = splitter.split_documents(docs)


# 2.5. Embedding Creation and Vector Store Indexing

In this cell, you will transform the text chunks created earlier into vector embeddings and store them in a vector database. These embeddings enable semantic search, allowing the system to retrieve relevant pieces of information based on meaning rather than exact word matches. You will also configure how many relevant chunks are retrieved for each query. This step forms the core “retrieval” component of the RAG pipeline.

* Initialize an embedding model suitable for semantic similarity search.

* Create and configure a vector store that indexes the document chunks produced in the previous cell.

* Add the chunked documents to the vector store.

* Set up a retrieval configuration (e.g., number of results k) and consider how this choice might affect answer quality.

In [228]:
embeddings = HuggingFaceEmbeddings(
   # TODO: 1. Initialize an embedding model for semantic search.
   model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    # TODO: 2. Create a vector store to index the document chunks.
    embedding_function = embeddings,
    collection_name="rag_docs"
)

# Add docs to vector store
print("Adding documents to vector store:")
vectorstore.add_documents(split_docs) # 3. Add the split documents to the vector store.

retriever = vectorstore.as_retriever(
    # TODO: 4. Set up a retrieval configuration.
    search_kwargs = {"k": 3 }
)

print("Vector store created successfully.")
print("Retriever configured to fetch top 3 documents")


Adding documents to vector store:
Vector store created successfully.
Retriever configured to fetch top 3 documents


# 2.6. Basic RAG

In this cell, you will implement a complete, end-to-end RAG pipeline. Given a user question, the system first retrieves semantically relevant document chunks from the vector store, then constructs a context from these chunks, and finally uses a large language model to generate an answer grounded in the retrieved information. This cell brings together all previous steps—retrieval, context construction, and generation—into a single working function.

In [229]:
def basic_rag_answer(question, k=5):
    # TODO: 1) Retrieve similar documents from the vector store
    retrieved_docs = vectorstore.similarity_search(question, k=k)

    # TODO: 2) Build a single context string from retrieved chunks
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])

    # TODO: 3) Construct a grounded prompt
    prompt = f"""Answer the question based ONLY on the following context.
If the answer is not in the context, say "I don't know."
    Context:
    {context}

    Question: {question}

    Answer:
    """

    # 4) Call the LLM to generate the answer
    return call_groq_llm(prompt)


In [230]:
print(basic_rag_answer("What is Self-RAG?"))


Self-RAG is an iterative refinement technique where the system retrieves documents, generates an answer, then refines the query based on that answer to retrieve better context in subsequent iterations.


# 2.7. End-to-End RAG Pipeline with Web Data

In this cell, you will construct a complete RAG pipeline using real-world web data.
The pipeline performs the following steps:

* Document loading from a live webpage

* Text chunking for efficient retrieval

* Embedding and vector storage using a dense vector database

* Retrieval of relevant document chunks for a query

* Grounded generation using a large language model

This cell demonstrates how RAG systems are engineered in practice, from raw data ingestion to final answer generation. Please implement the cells as instructed or write in the markdown for answering if needed.

In [231]:
# TODO:
# - Read through the imports and identify which ones are responsible for:
#   (1) document loading
#   (2) text chunking
#   (3) embeddings
#   (4) retrieval
#   (5) generation

import bs4
import os

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq


In [232]:
# TODO:
# - Replace the URL with another relevant article about RAG, agents, or LLM reasoning
# - (Optional) Add a second URL to compare retrieval behavior
print("Loading docs from the web...")
loader = WebBaseLoader(
    web_paths=(
        "https://lilianweng.github.io/posts/2023-06-23-agent/",
        # TODO: add another URL here
        "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    ),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()


# TODO:
# - Print the number of loaded documents
print(f"Number of loadaed documents:{len(docs)}")
# - Inspect the content of one document
print(f"\nFirst document preview (first 500 chars):\n{docs[0].page_content[:500]}")

Loading docs from the web...
Number of loadaed documents:2

First document preview (first 500 chars):


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [233]:
# TODO:
# - Experiment with different chunk_size and chunk_overlap values
# - Explain (in a markdown cell) why chunking is necessary for RAG

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # TODO: try 500 or 1500
    chunk_overlap=200     # TODO: try 50 or 300
)

splits = text_splitter.split_documents(docs)

# TODO:
# - Print the number of chunks created
print(f" Created {len(splits)} chunks from web documents")
# - Inspect the length of one chunk
print(f"\n Example chunk lengthL {len(splits[0].page_content)} characters")
print(f"Preview:\n {splits[0].page_content[:300]}...")


 Created 106 chunks from web documents

 Example chunk lengthL 969 characters
Preview:
 LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring...


In [234]:
# TODO:
# - Read about the embedding model used below
# - Explain why dense embeddings enable semantic search

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# create new vector store for web content
print("Creating vector store for web content")
vectorstore_web = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

# TODO:
# - Change k and observe how retrieval results differ
retriever = vectorstore_web.as_retriever(search_kwargs={"k":5})
print("web vector store created successfully!")


Creating vector store for web content
web vector store created successfully!


In [236]:
# TODO:
# - Modify the prompt to further reduce hallucinations
# - Add instructions such as:
#   “If the answer is not in the context, say 'I don't know'.”

prompt = ChatPromptTemplate.from_template(
    """You are an assistant that answers questions using ONLY the provided context.
If the answer is not in the context, say "I don't know based on the provided information."
Do not make up or infer information beyond what is explicitly stated.

Context:
{context}

Question:
{question}

Answer:
"""
)


In [237]:
# TODO:
# - Try a different Groq model
# - Experiment with temperature > 0 and observe changes

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


In [238]:
# TODO:
# - Explain why retrieved documents must be formatted before prompting
# - (Optional) Add separators or document numbering

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [239]:
# TODO:
# - Trace the data flow in this chain:
#   question → retrieval → formatting → prompt → LLM → output
# - Identify which component is responsible for retrieval

rag_chain = (
    {
        "context": retriever  | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully!")


RAG chain created successfully!


In [240]:
# Test with multiple questions
print("Question 1:")
print(rag_chain.invoke("What is Task Decomposition?"))

print("\n" + "="*50 + "\n")

print("Question 2:")
print(rag_chain.invoke("What are the different approaches to task decomposition?"))

# Failure case: asking about information not in the context
print("\n" + "="*50 + "\n")
print("Question 3 (Failure case - info not in docs):")
print(rag_chain.invoke("What is the population of Tokyo?"))

Question 1:
Task Decomposition is described in the context as a process that transforms big tasks into multiple manageable tasks. It is also mentioned as a technique that decomposes hard tasks into smaller and simpler steps.


Question 2:
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.


Question 3 (Failure case - info not in docs):
I don't know based on the provided information.


# 2.8. RRR-RAG (Rewrite–Retrieve–Respond)

This cell implements RRR-RAG (Rewrite–Retrieve–Respond), an extension of basic RAG.
Instead of directly retrieving documents using the user’s original question, the system first rewrites the question to make it more suitable for semantic search. The rewritten query is then used to retrieve relevant documents from the vector store, and finally the model generates an answer grounded only in the retrieved context.

The goal of this approach is to improve retrieval quality and, as a result, produce more accurate and relevant answers compared to standard RAG.

In [242]:
def rrr_rag_answer(question):
    """
    Rewrite–Retrieve–Respond RAG

    This function improves retrieval quality by first rewriting
    the user question, then retrieving documents using the rewritten
    query, and finally generating an answer grounded in the retrieved context.
    """

    # TODO 1: Write a rewrite prompt
    rewrite_prompt = f"""You are a query optimization expert. Rewrite the following question to make it more effective for semantic search in a document database.
Make the query more specific, clear, and focused on key concepts. Output ONLY the rewritten query without any explanation.

Original question: {question}

Rewritten query:"""

    # TODO 2: Call the Groq LLM to generate the rewritten query
    q_rewrite = call_groq_llm(rewrite_prompt).strip()

    # TODO 3: Retrieve relevant documents using the rewritten query
    docs = vectorstore.similarity_search(q_rewrite, k=5)

    # TODO 4: Build a single context string from the retrieved documents
    ctx = "\n\n".join([doc.page_content for doc in docs])

    # TODO 5: Construct the final answer prompt
    answer_prompt = f"""Answer the following question using ONLY the information provided in the context below.
If the answer cannot be found in the context, say "I don't have enough information to answer this question."

Context:
{ctx}

Question: {question}

Answer:"""

    # TODO 6: Call the Groq LLM to generate the final answer
    return call_groq_llm(answer_prompt)

In [246]:
print(rrr_rag_answer("Explain how refinement can improve RAG quality."))


I don't have enough information to answer this question.


# 2.9. Self-RAG

This cell implements Self-RAG, an iterative variant of Retrieval-Augmented Generation.
Instead of answering the question only once, the system repeatedly retrieves documents, generates an answer, and then refines the query based on the previous answer. Each iteration aims to improve retrieval quality, allowing the model to correct or enrich its understanding over time. This approach demonstrates how feedback loops can be used to improve RAG performance without fine-tuning the model.

In [248]:
def self_rag(question, iterations=2):
    """
    Self-RAG: Iterative Retrieval-Augmented Generation

    The system repeatedly:
    1) retrieves documents,
    2) generates an answer,
    3) refines the query based on the previous answer.
    """

    # TODO 1: Initialize the query used for the first retrieval step
    current_query = question

    # TODO 2: Initialize a variable to store the model's answer
    answer = ""

    for i in range(iterations):

        # TODO 3: Retrieval - Retrieve relevant documents using the current query
        docs = vectorstore.similarity_search(current_query, k=5)

        # TODO 4: Context construction - Combine retrieved documents into a single context string
        ctx = "\n\n".join([doc.page_content for doc in docs])

        # TODO 5: Answer generation - Build a prompt that uses ONLY the retrieved context
        prompt = f"""Answer the following question using ONLY the information provided in the context below.
Be specific and detailed. If the answer cannot be found in the context, say so.

Context:
{ctx}

Question: {question}

Answer:"""

        # TODO 6: Call the LLM to generate an answer
        answer = call_groq_llm(prompt)

        # TODO 7: Query refinement - Write a prompt that refines the current query
        refine_prompt = f"""Based on the answer below, generate a refined search query that would help retrieve more specific and relevant information to improve the answer.
Focus on aspects that need more detail or clarification. Output ONLY the refined query.

Original question: {question}
Current answer: {answer}

Refined query:"""

        # TODO 8: Generate the refined query using the LLM
        current_query = call_groq_llm(refine_prompt).strip()

    # TODO 9: Return the final answer after all iterations
    return answer

In [249]:
print(self_rag("How does RAG refine answers?", iterations=3))

The context does not mention how RAG refines answers. It only mentions that advanced techniques like Self-RAG and RRR improve answer quality using refinement loops, but it does not provide any information about how RAG refines answers.


In [250]:
questions = [
    "What is the idea of refinement loops in RAG?",
    "How does rewriting improve retrieval?"
]

for q in questions:
    print("="*80)
    print(f"QUESTION: {q}")
    print("="*80)
    print("\n[Basic RAG]")
    print(basic_rag_answer(q))
    print("\n[RRR RAG]")
    print(rrr_rag_answer(q))
    print("\n[Self-RAG]")
    print(self_rag(q))
    print("\n")

QUESTION: What is the idea of refinement loops in RAG?

[Basic RAG]
The idea of refinement loops in RAG (which is Self-RAG in this context) is that the system refines the query based on the answer generated in the previous iteration to retrieve better context in subsequent iterations.

[RRR RAG]
I don't have enough information to answer this question.

However, based on the context, I can infer that the refinement process is mentioned, but the term "refinement loops" is not explicitly mentioned.

[Self-RAG]
Unfortunately, the context does not provide a clear explanation of what refinement loops are in the context of RAG (Regressive Attention Generator). It only mentions that advanced techniques like Self-RAG and RRR improve answer quality using refinement loops, but does not provide a definition or description of refinement loops.


QUESTION: How does rewriting improve retrieval?

[Basic RAG]
Rewriting improves retrieval by making the user's question more search-friendly.

[RRR RAG]
I 

# 2.10. Comparison of RAG strategies

This cell compares three RAG strategies—Basic RAG, RRR-RAG (Rewrite–Retrieve–Respond), and Self-RAG—on the same set of questions. By running identical queries through different pipelines, you can observe how query rewriting and iterative refinement influence retrieval quality and the final generated answers. This comparison highlights the practical trade-offs between simplicity, robustness, and computational cost in RAG system design.

In [251]:
questions = [
    "What is the idea of refinement loops in RAG?",
    "How does rewriting improve retrieval?"
]

for q in questions:
    print("----")
    print("Basic RAG    :", basic_rag_answer(q))
    print("RRR RAG      :", rrr_rag_answer(q))
    print("Self-RAG     :", self_rag(q))


----
Basic RAG    : The idea of refinement loops in RAG is to refine the query based on the answer generated in the previous iteration to retrieve better context in subsequent iterations.
RRR RAG      : I don't have enough information to answer this question.
Self-RAG     : The context does not explicitly mention the term "refinement loops" in RAG. However, it does describe the process of refinement in Self-RAG as follows:

1. The system retrieves documents.
2. The system generates an answer.
3. The system refines the query based on that answer to retrieve better context in subsequent iterations.

This process is described as an "iterative refinement technique". This implies that the refinement process is repeated multiple times, but the context does not explicitly mention the term "refinement loops".
----
Basic RAG    : Rewriting the user's question to be more search-friendly.
RRR RAG      : I don't have enough information to answer this question.
Self-RAG     : The context does not m

***What You Should Remember:***

1. RAG combines information retrieval with language models so that answers are grounded in external knowledge rather than relying only on the model’s internal memory. This reduces hallucinations and improves factual accuracy.

2. Chroma Vector Store enables semantic retrieval by embedding text chunks into high-dimensional vector spaces. Queries are embedded in the same space, and the most semantically similar chunks are retrieved to provide relevant context.

3. BM25 Retriever performs lexical (keyword-based) retrieval, ranking documents by term frequency and inverse document frequency. It is especially effective when exact wording or terminology matters.

4. High-quality RAG systems often benefit from hybrid retrieval, where:

*  BM25 ensures precision through keyword matching, and

*  Vector retrieval (Chroma) ensures recall through semantic similarity.

5. Building a complete RAG pipeline requires:

*  Data preparation:
Collecting raw text (e.g., web pages, PDFs) and splitting it into manageable chunks that can be efficiently retrieved.

*  Retriever setup:
Creating one or more retrievers (vector-based, lexical, or hybrid) to fetch the most relevant context for a given query.

*  Prompt and context formatting:
Structuring retrieved documents into a clear context block that guides the language model to use only the provided information.

*  LLM integration:
Connecting the retriever output to a language model (e.g., Groq LLMs) that generates the final answer.

6. Advanced RAG techniques improve quality further:

*  RRR-RAG rewrites queries to improve retrieval quality before answering.

*  Self-RAG iteratively refines queries and answers using feedback loops.

7. There is a trade-off between simplicity and performance:

*  Basic RAG is fast and cost-effective.

*  Refinement-based RAG (RRR, Self-RAG) improves answer quality but requires additional LLM calls.

Congratulations! You've come to the end of this assignment.